In [1]:
! pip install nltk better_profanity

     |████████████████████████████████| 46 kB 3.9 MB/s eta 0:00:01


In [3]:
import re
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize

nltk.download('punkt')
nltk.download('punkt_tab')

# ---------- Load the dataset ----------
tickets_df = pd.read_csv("train.csv")

# See what columns actually exist in this dataset
print(tickets_df.columns.tolist())
print(tickets_df.head())

['count', 'hate_speech_count', 'offensive_language_count', 'neither_count', 'class', 'tweet']
   count  hate_speech_count  offensive_language_count  neither_count  class  \
0      3                  0                         0              3      2   
1      3                  0                         3              0      1   
2      3                  0                         3              0      1   
3      3                  0                         2              1      1   
4      6                  0                         6              0      1   

                                               tweet  
0  !!! RT @mayasolovely: As a woman you shouldn't...  
1  !!!!! RT @mleew17: boy dats cold...tyga dwn ba...  
2  !!!!!!! RT @UrKindOfBrand Dawg!!!! RT @80sbaby...  
3  !!!!!!!!! RT @C_G_Anderson: @viva_based she lo...  
4  !!!!!!!!!!!!! RT @ShenikaRoberts: The shit you...  


[nltk_data] Downloading package punkt to /home/computer/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/computer/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [6]:
import pandas as pd

tickets_df = pd.read_csv("train.csv")
print(tickets_df.columns.tolist())   # confirm what's really there

# Pick the first column that looks like it holds ticket text
text_col = [c for c in tickets_df.columns if any(k in c.lower() for k in ["text", "desc", "body", "subject", "message"])]
text_col = text_col[0] if text_col else tickets_df.columns[0]

ticket_text = tickets_df.loc[0, text_col]

print("Using column:", text_col)
print("\nRaw Ticket Text:\n", ticket_text)

['count', 'hate_speech_count', 'offensive_language_count', 'neither_count', 'class', 'tweet']
Using column: count

Raw Ticket Text:
 3


In [10]:
import re

# ---------- 2. Filtration ----------
def filter_text(text):
    text = str(text)
    text = re.sub(r'http\S+|www\S+|\S+@\S+', '', text)  # remove URLs/emails
    text = re.sub(r'[^\x00-\x7F]+', '', text)            # remove non-ASCII/emojis/encoding artifacts
    text = re.sub(r'[^\w\s]', ' ', text)                  # remove punctuation/special characters
    text = re.sub(r'\d+', ' ', text)                      # remove standalone numbers (e.g. order IDs)
    text = re.sub(r'\s+', ' ', text).strip()              # normalize whitespace
    return text

filtered_ticket = filter_text(ticket_text)

print("\nFiltered Ticket Text:\n", filtered_ticket)


Filtered Ticket Text:
 


In [12]:
# ---------- 3. Script Validation ----------
def is_valid_script(text, allowed_pattern=r'^[a-zA-Z0-9\s\.\,\!\?\-\@\&]+$'):
    """Checks whether each token belongs to the expected (Latin/English) script."""
    text = str(text)
    tokens = text.split()
    valid_tokens = [t for t in tokens if re.match(allowed_pattern, t)]
    invalid_tokens = [t for t in tokens if not re.match(allowed_pattern, t)]
    return valid_tokens, invalid_tokens

valid_tokens, invalid_tokens = is_valid_script(ticket_text)

print("\nNumber of Valid Script Tokens:", len(valid_tokens))
print("Number of Invalid Script Tokens:", len(invalid_tokens))
print("Sample Invalid Tokens:\n", invalid_tokens[:15])


Number of Valid Script Tokens: 1
Number of Invalid Script Tokens: 0
Sample Invalid Tokens:
 []


In [13]:
# ---------- 4. Toxic Word Detection & Filtration ----------
!pip install better_profanity

from better_profanity import profanity
profanity.load_censor_words()

def detect_toxic_words(text):
    words = str(text).split()
    return [w for w in words if profanity.contains_profanity(w)]

def filter_toxic_text(text):
    return profanity.censor(str(text))

toxic_words_found = detect_toxic_words(filtered_ticket)
censored_text = filter_toxic_text(filtered_ticket)

print("Filtered Text:\n", filtered_ticket)
print("\nDetected Toxic Words:\n", toxic_words_found)
print("\nCensored/Filtered Output:\n", censored_text)

Filtered Text:
 

Detected Toxic Words:
 []

Censored/Filtered Output:
 
